In [43]:
import sys, os
ROOT = "/Users/fserracrespi/Desktop/PD_PROJECT_UOFL" 
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
from utils.Decode import decoder_DF
from utils.dataframe_cols import cols_asignacion2
from utils.Prog_df import check_progression
from utils.Prog_df import progression_csv_upgrade
from utils.Prog_df import check_progression_multi
from utils.Prog_df import progression_multi_csv_upgrade
from utils.Prog_df import visit_csv_upgrade
from utils.Prog_df import check_visits
import json
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import kurtosis, skew
import re
from datetime import datetime
import math

In [44]:
path2='/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Study_Docs/Data___Databases/DATA/Data_Dictionary_-_Harmonized_12Sep2025.csv'
path3='/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Study_Docs/Data___Databases/DATA/Code_List_-_Harmonized_12Sep2025.csv'
code_cols=pd.read_csv(path2, dtype=str)
code_rows=pd.read_csv(path3, dtype=str)

# Initial Data Load

In [45]:
PATNOs = pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Relevant_presnece_PATNO', dtype=str)['PATNO'].tolist()
orden_visitas = ["BL", "V04", "V06", "V08", "V10", "V12"]
print(f'Total PATNOs to process: {len(PATNOs)}')

Total PATNOs to process: 1056


## Vital Signs

In [46]:
# Initial Data Load
vital_df=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Medical History/Medical/DATA/Vital_Signs_12Sep2025.csv', dtype=str)
vital_df=decoder_DF(vital_df, code_rows, code_cols, module='VITAL')
vital_df=vital_df[vital_df['PATNO'].isin(PATNOs)]
vital_df=vital_df.loc[vital_df['Visit ID'].isin(["BL", "V04", "V06", "V08", "V10", "V12"]),:]
vital_df['Visit ID'] = pd.Categorical(vital_df['Visit ID'], categories=orden_visitas, ordered=True)
vital_df=vital_df.sort_values(by=['PATNO', 'Visit ID']).reset_index(drop=True)                
print(f'Vital Signs records after filtering by PATNO: {len(vital_df)}')

Vital Signs records after filtering by PATNO: 3817


/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)


In [47]:
vital_relevant_cols = ['PATNO', 'Visit ID', 'Height (cm)', 'Weight (kg)','Temperature (Celsius)','Supine BP - systolic (mmHg)',
                    'Supine BP - diastolic (mmHg)','Supine heart rate (bpm)','Standing BP - systolic (mmHg)','Standing BP - diastolic (mmHg)','Standing heart rate (bpm)']

vital_df = vital_df[vital_relevant_cols]

info_height_weight = {}
patno_list = vital_df['PATNO'].unique().tolist()

for patno in patno_list:
    data_patno = vital_df[vital_df['PATNO'] == patno]
    info_height_weight[patno] = {'Height (cm)':[], 'Weight (kg)':[]}
    for index, row in data_patno.iterrows():
        height = row['Height (cm)']
        weight = row['Weight (kg)']
        info_height_weight[patno]['Height (cm)'].append(float(height))
        info_height_weight[patno]['Weight (kg)'].append(float(weight))
    

In [48]:
info_height_weight['100001']

{'Height (cm)': [188.0, 188.0, 188.0, 189.0, 188.0, 188.0],
 'Weight (kg)': [87.0, 88.0, 83.4, 86.0, 83.0, 85.1]}

In [49]:

info_height_weight = {}
patno_list = vital_df['PATNO'].unique().tolist()


for patno in patno_list:
    data_patno = vital_df[vital_df['PATNO'] == patno]
    info_height_weight[patno] = {'Height (cm)': [], 'Weight (kg)': []}
    for _, row in data_patno.iterrows():
        # Coerción segura a float (NaN si falla)
        height = pd.to_numeric(row['Height (cm)'], errors='coerce')
        weight = pd.to_numeric(row['Weight (kg)'], errors='coerce')
        info_height_weight[patno]['Height (cm)'].append(height)
        info_height_weight[patno]['Weight (kg)'].append(weight)

# --- Función para rellenar valores faltantes ---
def fill_missing_values(lista_val):
    lista_val = lista_val.copy()
    for i in range(1, len(lista_val)):
        if np.isnan(lista_val[i]):
            lista_val[i] = lista_val[i-1]
    for i in range(len(lista_val)-2, -1, -1):
        if np.isnan(lista_val[i]):
            lista_val[i] = lista_val[i+1]
    return lista_val

# --- Rellenar datos ---
for key in info_height_weight:
    info_height_weight[key]['Height (cm)'] = fill_missing_values(info_height_weight[key]['Height (cm)'])
    info_height_weight[key]['Weight (kg)'] = fill_missing_values(info_height_weight[key]['Weight (kg)'])

# --- Unir todas las listas ---
list_weight = []
list_height = []
list_keys = []

for key in info_height_weight:
    list_keys.append(key)
    list_weight += info_height_weight[key]['Weight (kg)']
    list_height += info_height_weight[key]['Height (cm)']


vital_df['Weight (kg)']=list_weight
vital_df['Height (cm)']=list_height




In [50]:
for col in vital_df.columns[2:]:
    vital_df[col] = pd.to_numeric(vital_df[col], errors='coerce')
    vital_df[col].fillna(vital_df[col].mean(), inplace=True)

vital_df.isna().sum()





/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_58251/1727225424.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  vital_df[col].fillna(vital_df[col].mean(), inplace=True)


PATNO                             0
Visit ID                          0
Height (cm)                       0
Weight (kg)                       0
Temperature (Celsius)             0
Supine BP - systolic (mmHg)       0
Supine BP - diastolic (mmHg)      0
Supine heart rate (bpm)           0
Standing BP - systolic (mmHg)     0
Standing BP - diastolic (mmHg)    0
Standing heart rate (bpm)         0
dtype: int64

In [51]:

# Orden correcto de visitas
visit_order = ["BL", "V04", "V06", "V08", "V10", "V12"]

# Asegurar categoría ordenada
vital_df["Visit ID"] = pd.Categorical(vital_df["Visit ID"], categories=visit_order, ordered=True)

# Lista única de pacientes
patients = vital_df["PATNO"].unique()

# Construir dataframe final vacío
df_list = []

for p in patients:
    # Extraer el subset del paciente
    sub = vital_df[vital_df["PATNO"] == p].copy()
    
    # Crear todas las visitas posibles para ese PATNO
    full_visits = pd.DataFrame({
        "PATNO": p,
        "Visit ID": visit_order
    })
    
    # Unir sin eliminar nada
    merged = full_visits.merge(sub, on=["PATNO", "Visit ID"], how="left")
    
    # Rellenar con forward fill por paciente
    merged = merged.sort_values("Visit ID").ffill()
    
    df_list.append(merged)

# Combinar todo
vital_df = pd.concat(df_list, ignore_index=True)

vital_df


,PATNO,Visit ID,Height (cm),Weight (kg),Temperature (Celsius),Supine BP - systolic (mmHg),Supine BP - diastolic (mmHg),Supine heart rate (bpm),Standing BP - systolic (mmHg),Standing BP - diastolic (mmHg),Standing heart rate (bpm)
0,100001,BL,188.0,87.0,36.0,113.0,72.0,65.0,106.0,78.0,86.0
1,100001,V04,188.0,88.0,35.7,115.0,67.0,64.0,110.0,73.0,76.0
2,100001,V06,188.0,83.4,36.4,113.0,69.0,89.0,94.0,65.0,80.0
3,100001,V08,189.0,86.0,37.0,123.0,74.0,72.0,113.0,79.0,90.0
4,100001,V10,188.0,83.0,36.5,104.0,59.0,72.0,96.0,62.0,99.0
...,...,...,...,...,...,...,...,...,...,...,...
6325,75562,V04,164.0,79.4,36.9,142.0,78.0,79.0,146.0,80.0,89.0
6326,75562,V06,164.0,79.4,36.9,142.0,78.0,79.0,146.0,80.0,89.0
6327,75562,V08,167.0,71.0,37.0,125.0,62.0,51.0,140.0,71.0,55.0
6328,75562,V10,168.0,70.4,35.0,116.0,58.0,61.0,126.0,55.0,62.0


In [52]:
print(vital_df['PATNO'].nunique())
print(vital_df.loc[vital_df['Visit ID']=='BL'].shape)
print(vital_df.loc[vital_df['Visit ID']=='V04'].shape)
print(vital_df.loc[vital_df['Visit ID']=='V06'].shape)
print(vital_df.loc[vital_df['Visit ID']=='V08'].shape)
print(vital_df.loc[vital_df['Visit ID']=='V10'].shape)
print(vital_df.loc[vital_df['Visit ID']=='V12'].shape)





1055
(1055, 11)
(1055, 11)
(1055, 11)
(1055, 11)
(1055, 11)
(1055, 11)


## LEDD_Concomitant_Medication_Log

In [53]:
LEDD_df=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Medical History/Medical/DATA/LEDD_Concomitant_Medication_Log_12Sep2025.csv', dtype=str)
LEDD_df=decoder_DF(LEDD_df, code_rows, code_cols, module='LEDDLOG')
LEDD_df.drop(columns=['Page Name','Total Daily Dose Administered - if Liquid, Infusion, or Injection'], inplace=True)
LEDD_df['Start Date']=pd.to_datetime(LEDD_df['Start Date'], errors='coerce')
LEDD_df['Stop Date']=pd.to_datetime(LEDD_df['Stop Date'], errors='coerce')
hoy = pd.Timestamp.today().normalize()   # sin hora, solo fecha
LEDD_df['Stop Date']  = LEDD_df['Stop Date'].fillna(hoy)
LEDD_df=LEDD_df[LEDD_df['PATNO'].isin(PATNOs)]
LEDD_df

/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)
/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_58251/2302690581.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  LEDD_df['Start Date']=pd.to_datetime(LEDD_df['Start Date'], errors='coerce')
/var/folders/vr/_v9wq92941bflv75

,Record ID,PATNO,Visit ID,Medication,Dose in mg of the Levodopa equivalent drug only,Dose Strength,Dose Taken,Frequency,Start Date,Stop Date,Levodopa Equivalent Daily Dose,Date of original data entry,Date of most recent update to record
0,IA98224,3001,ED,Amantadine,100.000,100 mg PO,1.00,1,2022-03-01,2022-07-01,100.0000000000000000,09/2022,2023-11-14 00:00:00.0
1,IA98222,3001,ED,Carbidopa/Levodopa IR,100.000,25/100 mg PO,1.00,7,2020-07-01,2026-01-16,700.0000000000000000,01/2022,2023-11-14 00:00:00.0
2,IA98223,3001,ED,Amantadine,100.000,100 mg PO,1.00,2,2020-07-01,2022-03-01,200.0000000000000000,07/2022,2023-11-14 00:00:00.0
3,IA248838,3001,ED,Carbidopa/Levodopa CR,100.000,25/100 mg PO,2.00,1,2023-02-01,2024-01-01,150.0000000000000000,03/2023,2024-09-25 00:00:00.0
4,516506501,3001,LOG,SINEMET 25/100,NaN,NaN,NaN,NaN,2015-03-01,2016-03-01,400.00,04/2015,2020-06-25 16:04:31.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8880,IA744997,466157,ED,Pramipexole IR,1.100,1.1 mg PO,1.00,1,2019-07-01,2026-01-16,110.0000000000000000,05/2025,2025-05-21 00:00:00.0
8881,IA792686,476236,ED,Carbidopa/Levodopa CR,100.000,25/100 mg PO,1.00,1,2024-11-01,2026-01-16,75.0000000000000000,07/2025,2025-07-25 00:00:00.0
8882,IA792688,476236,ED,Pramipexole ER,1.570,1.57 mg PO,1.00,1,2024-11-01,2026-01-16,157.0000000000000000,07/2025,2025-07-25 00:00:00.0
8883,IA792684,476236,ED,Co-careldopa (Carbidopa/Levodopa IR),100.000,25/100 mg PO,1.00,3,2024-11-01,2026-01-16,300.0000000000000000,07/2025,2025-07-25 00:00:00.0


In [54]:
LEDD_df['Medication'] = LEDD_df['Medication'].apply(lambda x: x.lower())
# elimina solo paréntesis y comas
LEDD_df['Medication'] = LEDD_df['Medication'].apply(
    lambda x: re.sub(r'[(),]', '', x)
)
LEDD_df['Medication'] = LEDD_df['Medication'].apply(
    lambda x: re.sub(r'([A-Za-z]+)(\d+)', r'\1 \2', x)
)

LEDD_df['Medication'] = LEDD_df['Medication'].apply(lambda x: re.sub(r'(?<!\d)[/\-](?!\d)', ' ', x))

# BAG OF WORDS PARA MEDICACIONES
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()

# Ajustar y transformar
bow = vectorizer.fit_transform(LEDD_df['Medication'])

# Convertir a DataFrame legible
bow_df = pd.DataFrame(bow.toarray(), columns=vectorizer.get_feature_names_out())

col_list=[]
count_list=[]
for col in bow_df.columns:
    col_list.append(col)
    count_list.append(bow_df[col].sum())

df_bag_of_words=pd.DataFrame({'Medication':col_list, 'Count':count_list})
df_bag_of_words.sort_values(by='Count', ascending=False, inplace=True)
# no queremos numericos en la columan no  tiene sentido de medicamentos
df_bag_of_words = df_bag_of_words[~df_bag_of_words['Medication'].str.contains(r'\d')]

# df_bag_of_words.to_csv('/Users/fserracrespi/Desktop/LEDD_BAG_OF_WORDS.csv', index=False)

# problemas del bag of words
# - formass de escribir el mismo medicamento pero diferente (levodopa, levo dopa, l-dopa, l dopa, etc)
# - introduccion de marcas que estios presentan sucompuestos sinemet es una marca que contiene levodopa y carbidopa


In [55]:
# Primera fase: medicamentos brand a estado generico 
medicamentos = {
    "levodopa carbidopa": [
        "sinemet", "sinement", "sinamet", "sinement",
        "rytary", "levocomp", "careldopa",
        "levobeta", "crexont", "duodopa", "isicom",'dopicar'
    ],
    "levodopa benserazide": [
        "madopar", "prolopa"
    ],
    "rasagiline": [
        "azilect", "mesylate"
    ],
    "ropinirole": [
        "requip"
    ],
    "pramipexole": [
        "mirapex", "sifrol", "oprymea", "mirapexin"
    ],
    "rotigotine": [
        "neupro", "nupro"
    ],
    "levodopa carbidopa entacapone": [
        "stalevo"
    ],
    "safinamide": [
        "xadago"
    ],
    "amantadine": [
        "gocovri", "merz", "symmetrel", "osmolex"
    ],
    "piribedil": [
        "clarium"
    ],
    "levodopa": [
        "inbrija", "neuraxpharm"
    ],
    "trihexyphenidyl": [
        "artane"
    ],
    "entacapone": [
        "comtan"
    ],
    "foscarbidopa foslevodopa": [
        "vyalev"
    ],
    "istradefylline": [
        "nourianz"
    ],
    "apomorphine": [
        "apokyn", "kynmobi"
    ],
    "melevodopa carbidopa": [
        "sirio"
    ],
    "selegiline": [
        "eldepryl"
    ],
    'entacapone': ['comtess'],
    'zonisamide':['zonegran'],
    'opicapone':['ongentys']
}

def map_to_generic(medication_name):
    
    list=medication_name.split()
    for word in list:
        for key in medicamentos:
            if word in medicamentos[key]:
                generic_med=key
                return generic_med

LEDD_df['Generic_Medication'] = LEDD_df['Medication'].apply(map_to_generic)
LEDD_df[['Medication', 'Generic_Medication']].head(10)

# fase 2 : mapeo de errores tipograficos y variaciones en la escritura

normalizacion = {
    "levodopa": [
        "levodopa", "levodop", "levodapa", "levopda", "loevadopa",
        "levopdopa", "levodpoav", "levodpa", "ledopoda", "ldopa",
        "levadopa", "levidopa", "dopa", "levopar"
    ],
    "carbidopa": [
        "carbidopa", "carbodipa", "cabidopa", "carbodopa",
        "caridopa", "carpidopa", "carvidopa", "carb", "carbdiopa"
    ],
    "amantadine": [
        "amantadine", "amantadin", "amandatine"
    ],
    "pramipexole": [
        "pramipexole", "pramiprexole", "pramipexolo",
        "pramiplexole", "prampexole", "pramipexol"
    ],
    "benserazide": [
        "benserazide", "benserazid", "benzerazide",
        "benseracide", "benzerzide", "benseracid", "benserzid"
    ],
    "rasagiline": [
        "rasagiline", "rasagilina", "rasagaline", "rasagilin",
        "rasageline", "rasagalin", "rasigaline", "rasagilene"
    ],
    "ropinirole": [
        "ropinirole", "ropinerole", "ropinirol", "ropirinole",
        "ropinrole"
    ],
    "entacapone": [
        "entacapone", "entacapona", "entacopone",
        "entcapone", "entacaprone", "entacapon"
    ],
    "rotigotine": [
        "rotigotine", "rotigotin", "rotigine"
    ],
    "selegiline": [
        "selegiline", "selegeline", "selegilene", "selegelina"
    ],
    "safinamide": [
        "safinamide", "safinamida"
    ],
    "opicapone": [
        "opicapone", "opicapon"
    ],
    "trihexyphenidyl": [
        "trihexyphenidyl", "trihexphenidyl",
        "trihexyphenidate", "trihexyphenidol"
    ],
    "piribedil": [
        "piribedil"
    ],
    "istradefylline": [
        "istradefylline"
    ],
    "apomorphine": [
        "apomorphine", "apo",'apomorfin'
    ],
    "foscarbidopa": [
        "foscarbidopa"
    ],
    "foslevodopa": [
        "foslevodopa"
    ],
    "melevodopa": [
        "melevodopa"
    ],
    "dihydrochloride": [
        "dihydrochloride"
    ],
    "benztropine": [
        "benztropine"
    ],
    'tolcapone': ['tolcapone']
    
}

def correccion_topografica(medication_name):
    list_final=[]
    list=medication_name.split()
    for word in list:
        for key in normalizacion:
            if word in normalizacion[key]:
                list_final.append(key)
                
    med_word=''
    for word in list_final:
        med_word=med_word+' '+word
    med_word=med_word.lstrip()
    return med_word  

LEDD_df['Corrected_Medication'] = LEDD_df['Medication'].apply(correccion_topografica)
LEDD_df[['Medication','Generic_Medication','Corrected_Medication']].head(10)

def combinar_final(generic, corrected):
    if pd.isna(generic):
        generic = ""
    if pd.isna(corrected):
        corrected = ""

    palabras = set((generic + " " + corrected).split())
    return " ".join(sorted(palabras))

LEDD_df["Final_Med"] = LEDD_df.apply(
    lambda row: combinar_final(row["Generic_Medication"], row["Corrected_Medication"]), axis=1
)
LEDD_df.drop(columns=['Generic_Medication','Corrected_Medication'], inplace=True)

LEDD_df.head(3)




,Record ID,PATNO,Visit ID,Medication,Dose in mg of the Levodopa equivalent drug only,Dose Strength,Dose Taken,Frequency,Start Date,Stop Date,Levodopa Equivalent Daily Dose,Date of original data entry,Date of most recent update to record,Final_Med
0,IA98224,3001,ED,amantadine,100.000,100 mg PO,1.00,1,2022-03-01,2022-07-01,100.0000000000000000,09/2022,2023-11-14 00:00:00.0,amantadine
1,IA98222,3001,ED,carbidopa levodopa ir,100.000,25/100 mg PO,1.00,7,2020-07-01,2026-01-16,700.0000000000000000,01/2022,2023-11-14 00:00:00.0,carbidopa levodopa
2,IA98223,3001,ED,amantadine,100.000,100 mg PO,1.00,2,2020-07-01,2022-03-01,200.0000000000000000,07/2022,2023-11-14 00:00:00.0,amantadine


In [56]:
# n0 de meds
LEDD_df['Med_count']=LEDD_df['Final_Med'].apply(lambda x: len(x.split()))
LEDD_1_meds=LEDD_df[LEDD_df['Med_count']==1]
LEDD_2_meds=LEDD_df[LEDD_df['Med_count']==2]
LEDD_3_meds=LEDD_df[LEDD_df['Med_count']==3]


print(f'Número de registros con 1 medicamento: {len(LEDD_1_meds)}')
print(f'Número de registros con 2 medicamentos: {len(LEDD_2_meds)}')
print(f'Número de registros con 3 medicamentos: {len(LEDD_3_meds)}')



Número de registros con 1 medicamento: 2729
Número de registros con 2 medicamentos: 3042
Número de registros con 3 medicamentos: 142


#### LEDD 1

In [57]:
LEDD_1_meds["LEDD_num"] = pd.to_numeric(
    LEDD_1_meds["Levodopa Equivalent Daily Dose"], errors="coerce"
)

# eliminar filas donde la columna no era numérica
LEDD_1_meds = LEDD_1_meds.dropna(subset=["LEDD_num"])



/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_58251/966326823.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  LEDD_1_meds["LEDD_num"] = pd.to_numeric(


In [58]:
# Crear df_counts
df_counts = (
    LEDD_1_meds
    .groupby('Final_Med')['Dose Strength']
    .value_counts()
    .reset_index(name='count')
)

# Extraer número de dosis
df_counts['dose_num'] = df_counts['Dose Strength'].str.extract(r'([\d\.]+)').astype(float)

# dict type medication
meds_type={
 'amantadine': 'PO',
 'apomorphine': 'SUBLINGUAL',
 'benztropine': 'PO',
 'entacapone': 'PO',
 'istradefylline': 'PO',
 'levodopa': 'INHALED',
 'opicapone': 'PO',
 'piribedil': 'PO',
 'pramipexole': 'PO',
 'rasagiline': 'PO',
 'ropinirole': 'PO',
 'rotigotine': 'TRANSDERMAL',
 'safinamide': 'PO',
 'selegiline': 'PO',
 'tolcapone': 'PO',
 'trihexyphenidyl': 'PO',
 'zonisamide': 'PO'
}

# Media ponderada
weighted_means = (
    df_counts
    .groupby('Final_Med')
    .apply(lambda g: (g['dose_num'] * g['count']).sum() / g['count'].sum())
    .reset_index(name='weighted_mean')
)

# --- Crear columna con dosis numérica original ---
LEDD_1_meds["Dose Strength Number"] = (
    LEDD_1_meds["Dose Strength"]
    .str.extract(r'([\d\.]+)')
    .astype(float)
)

# --- Rellenar NaN usando las medias por medicamento ---
LEDD_1_meds = LEDD_1_meds.merge(weighted_means, on="Final_Med", how="left")

LEDD_1_meds["Dose Strength Number"] = (
    LEDD_1_meds["Dose Strength Number"]
    .fillna(LEDD_1_meds["weighted_mean"])
)

# Quitar columna auxiliar si no la quieres
LEDD_1_meds.drop(columns=["weighted_mean"], inplace=True)

LEDD_1_meds['Medication_Type'] = LEDD_1_meds['Final_Med'].map(meds_type)
LEDD_1_meds.drop(columns=['Dose in mg of the Levodopa equivalent drug only','Dose Strength','Dose Taken'],inplace=True)
LEDD_1_meds= LEDD_1_meds.loc[~LEDD_1_meds['Medication'].isin(['melevodopa','carbidopa 25/ldopa 100','isicomlevodopa  carbdiopa 50/12.5']), :]
LEDD_1_meds.isna().sum()

/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_58251/3485308015.py:37: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: (g['dose_num'] * g['count']).sum() / g['count'].sum())


Record ID                                  0
PATNO                                      0
Visit ID                                   0
Medication                                 0
Frequency                               1382
Start Date                                 0
Stop Date                                  0
Levodopa Equivalent Daily Dose             0
Date of original data entry                6
Date of most recent update to record       0
Final_Med                                  0
Med_count                                  0
LEDD_num                                   0
Dose Strength Number                       0
Medication_Type                            0
dtype: int64

In [59]:
LEDD_1_meds=LEDD_1_meds[['PATNO','Visit ID','Final_Med','Medication_Type','Dose Strength Number','Frequency','Levodopa Equivalent Daily Dose','Start Date','Stop Date']]
LEDD_1_meds['Dose Strength Number']=LEDD_1_meds['Dose Strength Number'].round(0)
LEDD_1_meds['Levodopa Equivalent Daily Dose']=LEDD_1_meds['Levodopa Equivalent Daily Dose'].astype(float).round(0)

LEDD_1_meds['Frequency'] = (
    LEDD_1_meds['Levodopa Equivalent Daily Dose'] /
    LEDD_1_meds['Dose Strength Number']
).round(0)
LEDD_1_meds

LEDD_1_meds.columns=['PATNO','Visit ID','Medication','Medication_Type','Dose_Strength_mg','Frequency_per_day','LEDD_mg_per_day','Start Date','Stop Date']


In [60]:
med_list=['amantadine', 'selegiline', 'rotigotine', 'pramipexole',
       'ropinirole', 'rasagiline', 'levodopa', 'zonisamide',
       'apomorphine', 'trihexyphenidyl', 'safinamide', 'piribedil']

col1='Dose_Strength_mg_'
col2='Total_mg_per_day_'

for col in med_list:
       new_col_dose=col1+col
       new_col_total=col2+col
       LEDD_1_meds[new_col_dose]=0
       LEDD_1_meds[new_col_total]=0


for idx,row in LEDD_1_meds.iterrows():
       med=row['Medication']
       LEDD_1_meds.loc[idx,col1+med]=row['Dose_Strength_mg']
       LEDD_1_meds.loc[idx,col2+med]=row['Dose_Strength_mg']*row['Frequency_per_day']

LEDD_1_meds.drop(columns=['Medication_Type','Dose_Strength_mg','LEDD_mg_per_day'],inplace=True)
LEDD_1_meds
       

,PATNO,Visit ID,Medication,Frequency_per_day,Start Date,Stop Date,Dose_Strength_mg_amantadine,Total_mg_per_day_amantadine,Dose_Strength_mg_selegiline,Total_mg_per_day_selegiline,...,Dose_Strength_mg_zonisamide,Total_mg_per_day_zonisamide,Dose_Strength_mg_apomorphine,Total_mg_per_day_apomorphine,Dose_Strength_mg_trihexyphenidyl,Total_mg_per_day_trihexyphenidyl,Dose_Strength_mg_safinamide,Total_mg_per_day_safinamide,Dose_Strength_mg_piribedil,Total_mg_per_day_piribedil
0,3001,ED,amantadine,1.0,2022-03-01,2022-07-01,100,100,0,0,...,0,0,0,0,0,0,0,0,0,0
1,3001,ED,amantadine,2.0,2020-07-01,2022-03-01,100,200,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3001,LOG,selegiline,10.0,2012-07-01,2012-08-01,0,0,5,50,...,0,0,0,0,0,0,0,0,0,0
3,3001,LOG,rotigotine,12.0,2013-08-01,2013-09-01,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,3001,LOG,amantadine,2.0,2018-07-01,2020-06-01,105,210,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2471,398222,ED,rasagiline,100.0,2020-01-01,2026-01-16,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2472,398222,ED,amantadine,1.0,2022-01-01,2026-01-16,100,100,0,0,...,0,0,0,0,0,0,0,0,0,0
2473,398222,ED,pramipexole,105.0,2020-01-01,2026-01-16,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2474,466157,ED,pramipexole,110.0,2019-07-01,2026-01-16,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


#### LEDD 2

In [61]:
LEDD_2_meds=LEDD_2_meds.loc[LEDD_2_meds['Final_Med'].isin(['carbidopa levodopa','benserazide levodopa']), :]
LEDD_2_meds

,Record ID,PATNO,Visit ID,Medication,Dose in mg of the Levodopa equivalent drug only,Dose Strength,Dose Taken,Frequency,Start Date,Stop Date,Levodopa Equivalent Daily Dose,Date of original data entry,Date of most recent update to record,Final_Med,Med_count
1,IA98222,3001,ED,carbidopa levodopa ir,100.000,25/100 mg PO,1.00,7,2020-07-01,2026-01-16,700.0000000000000000,01/2022,2023-11-14 00:00:00.0,carbidopa levodopa,2
3,IA248838,3001,ED,carbidopa levodopa cr,100.000,25/100 mg PO,2.00,1,2023-02-01,2024-01-01,150.0000000000000000,03/2023,2024-09-25 00:00:00.0,carbidopa levodopa,2
4,516506501,3001,LOG,sinemet 25/100,NaN,NaN,NaN,NaN,2015-03-01,2016-03-01,400.00,04/2015,2020-06-25 16:04:31.0,carbidopa levodopa,2
8,559553601,3001,LOG,sinemet 25/100,NaN,NaN,NaN,NaN,2016-03-01,2020-06-01,600.00,03/2016,2022-09-13 07:29:02.0,carbidopa levodopa,2
10,501974201,3001,LOG,sinemet 25/100,NaN,NaN,NaN,NaN,2014-12-01,2015-03-01,300.00,02/2015,2020-06-25 16:04:31.0,carbidopa levodopa,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8873,IA748124,446733,ED,sinemet carbidopa levodopa ir,100.000,25/100 mg PO,0.50,4,2020-08-01,2026-01-16,200.0000000000000000,05/2025,2025-05-28 00:00:00.0,carbidopa levodopa,2
8879,IA744998,466157,ED,madopar levodopa benserazide ir,200.000,200/50 mg PO,0.50,5,2024-11-01,2026-01-16,500.0000000000000000,05/2025,2025-05-21 00:00:00.0,benserazide levodopa,2
8881,IA792686,476236,ED,carbidopa levodopa cr,100.000,25/100 mg PO,1.00,1,2024-11-01,2026-01-16,75.0000000000000000,07/2025,2025-07-25 00:00:00.0,carbidopa levodopa,2
8883,IA792684,476236,ED,co careldopa carbidopa levodopa ir,100.000,25/100 mg PO,1.00,3,2024-11-01,2026-01-16,300.0000000000000000,07/2025,2025-07-25 00:00:00.0,carbidopa levodopa,2


In [62]:
# 1) Mask que detecta números con separador / o -
mask = LEDD_2_meds["Medication"].str.contains(
    r'\d+\.?\d*[/-]\d+\.?\d*', 
    na=False
) & LEDD_2_meds['Dose Strength'].isna()

# 2) Asignar Dose Strength desde Medication
LEDD_2_meds.loc[mask, 'Dose Strength'] = (
    LEDD_2_meds.loc[mask, 'Medication']
    .str.extract(r'(\d+\.?\d*[/-]\d+\.?\d*)')[0]
)

# 3) Extraer Dose1 y Dose2 permitiendo / o -
LEDD_2_meds[['Dose1', 'Dose2']] = (
    LEDD_2_meds['Dose Strength']
    .str.extract(r'([\d\.]+)\s*[/-]\s*([\d\.]+)')
    .astype(float)
)

# 4) Guardar la fracción completa (25/100 o 25-100)
LEDD_2_meds['Dose_fraction'] = (
    LEDD_2_meds['Dose Strength']
    .str.extract(r'(\d+\.?\d*[/-]\d+\.?\d*)')[0]
)

# 5) Agrupar por Medication + Final_Med
LEDD_2_meds.groupby(
    ['Medication','Final_Med']
)['Dose_fraction'].value_counts(dropna=False).reset_index(name='count') \
.to_csv('/Users/fserracrespi/Desktop/LEDD_2_meds_dose_strength_counts.csv', index=False)


In [63]:
LEDD_2_meds['Levodopa_Dose'] = LEDD_2_meds.apply(lambda row: row['Dose1'] if row['Dose1']>row['Dose2'] else row['Dose2'], axis=1)
LEDD_2_meds['Carbidopa_Benserazide_Dose'] = LEDD_2_meds.apply(lambda row: row['Dose1'] if row['Dose1']<row['Dose2'] else row['Dose2'], axis=1)
LEDD_2_meds=LEDD_2_meds[['PATNO','Visit ID','Final_Med','Dose_fraction','Levodopa_Dose','Carbidopa_Benserazide_Dose','Frequency','Levodopa Equivalent Daily Dose','Start Date','Stop Date']]
LEDD_2_meds.isna().sum()



PATNO                                0
Visit ID                             0
Final_Med                            0
Dose_fraction                      235
Levodopa_Dose                      235
Carbidopa_Benserazide_Dose         235
Frequency                         1025
Levodopa Equivalent Daily Dose      35
Start Date                           0
Stop Date                            0
dtype: int64

In [64]:
led_filter = LEDD_2_meds.loc[LEDD_2_meds['Final_Med'].isin(['carbidopa levodopa']), :]


In [65]:

mean_vals = (
    LEDD_2_meds
    .groupby('Final_Med')[['Levodopa_Dose', 'Carbidopa_Benserazide_Dose']]
    .mean()
    .round(0)
)

for idx, row in LEDD_2_meds.iterrows():
    if pd.isna(row['Dose_fraction']):
        med = row['Final_Med']
        LEDD_2_meds.loc[idx, 'Levodopa_Dose'] = mean_vals.loc[med, 'Levodopa_Dose']
        LEDD_2_meds.loc[idx, 'Carbidopa_Benserazide_Dose'] = mean_vals.loc[med, 'Carbidopa_Benserazide_Dose']




cols = ['Levodopa_Dose', 'Carbidopa_Benserazide_Dose', 'Frequency', 'Levodopa Equivalent Daily Dose']
LEDD_2_meds[cols] = LEDD_2_meds[cols].apply(pd.to_numeric, errors='coerce')

# ---- SEGUNDA FASE: CÁLCULO LEDD Y FREQUENCY ----
for idx, row in LEDD_2_meds.iterrows():

    # CASO 1: calcular LEDD
    if pd.notna(row['Frequency']) and pd.notna(row['Levodopa_Dose']):
        LEDD_2_meds.at[idx, 'Levodopa Equivalent Daily Dose'] = (
            row['Levodopa_Dose'] * row['Frequency']
        )

    # CASO 2: calcular Frequency
    if pd.isna(row['Frequency']) and pd.notna(row['Levodopa Equivalent Daily Dose']) and pd.notna(row['Levodopa_Dose']):
        LEDD_2_meds.at[idx, 'Frequency'] = round(
            row['Levodopa Equivalent Daily Dose'] / row['Levodopa_Dose']
        )


LEDD_2_meds.drop(columns=['Dose_fraction'], inplace=True)
LEDD_2_meds.dropna(subset=['Frequency','Levodopa Equivalent Daily Dose'], inplace=True)

LEDD_2_meds['Dose_Strength_mg_carbidopa'] = LEDD_2_meds.apply(
    lambda r: r['Carbidopa_Benserazide_Dose'] 
              if 'carbidopa' in r['Final_Med'].lower() else 0,
    axis=1
)

LEDD_2_meds['Dose_Strength_mg_benserazide'] = LEDD_2_meds.apply(
    lambda r: r['Carbidopa_Benserazide_Dose'] 
              if 'benserazide' in r['Final_Med'].lower() else 0,
    axis=1
)

LEDD_2_meds['Total_mg_per_day_carbidopa']= LEDD_2_meds['Dose_Strength_mg_carbidopa'] * LEDD_2_meds['Frequency']
LEDD_2_meds['Total_mg_per_day_benserazide']= LEDD_2_meds['Dose_Strength_mg_benserazide'] * LEDD_2_meds['Frequency']
LEDD_2_meds['Total_mg_per_day_levodopa']= LEDD_2_meds['Levodopa_Dose'] * LEDD_2_meds['Frequency']

LEDD_2_meds.rename(columns={'Levodopa_Dose':'Dose_Strength_mg_levodopa'}, inplace=True)
LEDD_2_meds.drop(columns=['Carbidopa_Benserazide_Dose','Levodopa Equivalent Daily Dose'], inplace=True)
LEDD_2_meds.rename(columns={'Frequency':'Frequency_per_day','Final_Med':'Medication'}, inplace=True)
LEDD_2_meds


/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_58251/1253428738.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  LEDD_2_meds[cols] = LEDD_2_meds[cols].apply(pd.to_numeric, errors='coerce')
/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_58251/1253428738.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  LEDD_2_meds.drop(columns=['Dose_fraction'], inplace=True)
/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_58251/1253428738.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

Se

,PATNO,Visit ID,Medication,Dose_Strength_mg_levodopa,Frequency_per_day,Start Date,Stop Date,Dose_Strength_mg_carbidopa,Dose_Strength_mg_benserazide,Total_mg_per_day_carbidopa,Total_mg_per_day_benserazide,Total_mg_per_day_levodopa
1,3001,ED,carbidopa levodopa,100.0,7.0,2020-07-01,2026-01-16,25.0,0.0,175.0,0.0,700.0
3,3001,ED,carbidopa levodopa,100.0,1.0,2023-02-01,2024-01-01,25.0,0.0,25.0,0.0,100.0
4,3001,LOG,carbidopa levodopa,100.0,4.0,2015-03-01,2016-03-01,25.0,0.0,100.0,0.0,400.0
8,3001,LOG,carbidopa levodopa,100.0,6.0,2016-03-01,2020-06-01,25.0,0.0,150.0,0.0,600.0
10,3001,LOG,carbidopa levodopa,100.0,3.0,2014-12-01,2015-03-01,25.0,0.0,75.0,0.0,300.0
...,...,...,...,...,...,...,...,...,...,...,...,...
8873,446733,ED,carbidopa levodopa,100.0,4.0,2020-08-01,2026-01-16,25.0,0.0,100.0,0.0,400.0
8879,466157,ED,benserazide levodopa,200.0,5.0,2024-11-01,2026-01-16,0.0,50.0,0.0,250.0,1000.0
8881,476236,ED,carbidopa levodopa,100.0,1.0,2024-11-01,2026-01-16,25.0,0.0,25.0,0.0,100.0
8883,476236,ED,carbidopa levodopa,100.0,3.0,2024-11-01,2026-01-16,25.0,0.0,75.0,0.0,300.0


#### LEDD 3

In [66]:
list_nonumeric_dose=[]
for idx, row in LEDD_3_meds.iterrows():
    if pd.isna(row['Dose Strength']):
        if re.search(r"\d", row['Medication']):
            # print(row['Medication'])
            pass
        else:
            list_nonumeric_dose.append(row['Medication'])
    
set_nonumeric_dose=set(list_nonumeric_dose)
set_nonumeric_dose

LEDD_3_meds=LEDD_3_meds.loc[~LEDD_3_meds['Medication'].isin(set_nonumeric_dose), :]

LEDD_3_meds['Medication'].unique()

stalevo_dict={'stalevo 100':'25/100/200',
                'stalevo 125':'31.25/125/200',
                'stalevo 150':'37.5/150/200',
                'stalevo 175':'43.275/175/200',
                'stalevo 200':'50/200/200',
                'stalevo 25':'6/50/200',
                'stalevo 50':'12.5/50/200',
                'stalevo 75':'19/50/200'
                }

for idx, row in LEDD_3_meds.iterrows():
    if pd.isna(row['Dose Strength']):
        if re.search(r"stalevo \d", row['Medication']):
            match = re.search(r"(stalevo\s*\d+)", row['Medication'], flags=re.IGNORECASE)
            valor = stalevo_dict[match.group(1)]
            LEDD_3_meds.at[idx, 'Dose Strength'] = valor
            
LEDD_3_meds=LEDD_3_meds[['PATNO', 'Visit ID','Final_Med', 'Dose Strength', 'Frequency','Start Date', 'Stop Date']]     
LEDD_3_meds.dropna(subset=['Dose Strength','Frequency'], inplace=True)






In [67]:
LEDD_3_meds[['Dose_Strength_mg_carbidopa', 'Dose_Strength_mg_levodopa','Dose_Strength_mg_entacapone']] = (
    LEDD_3_meds['Dose Strength']
    .str.extract(r'([\d\.]+)\s*[/-]\s*([\d\.]+)*[/-]\s*([\d\.]+)')
    .astype(float)
)



LEDD_3_meds['Total_mg_per_day_carbidopa'] = LEDD_3_meds['Dose_Strength_mg_carbidopa'] * LEDD_3_meds['Frequency'].astype(float)
LEDD_3_meds['Total_mg_per_day_levodopa'] = LEDD_3_meds['Dose_Strength_mg_levodopa'] * LEDD_3_meds['Frequency'] .astype(float)
LEDD_3_meds['Total_mg_per_day_entacapone'] = LEDD_3_meds['Dose_Strength_mg_entacapone'] * LEDD_3_meds['Frequency'].astype(float) 
LEDD_3_meds.drop(columns=['Dose Strength'], inplace=True)
LEDD_3_meds.rename(columns={'Frequency':'Frequency_per_day','Final_Med':'Medication'}, inplace=True)
LEDD_3_meds


,PATNO,Visit ID,Medication,Frequency_per_day,Start Date,Stop Date,Dose_Strength_mg_carbidopa,Dose_Strength_mg_levodopa,Dose_Strength_mg_entacapone,Total_mg_per_day_carbidopa,Total_mg_per_day_levodopa,Total_mg_per_day_entacapone
754,3173,ED,carbidopa entacapone levodopa,2,2024-07-01,2026-01-16,25.00,100.0,200.0,50.00,200.0,400.0
1523,3383,ED,carbidopa entacapone levodopa,6,2024-04-01,2026-01-16,25.00,100.0,200.0,150.00,600.0,1200.0
1524,3383,ED,carbidopa entacapone levodopa,5,2021-11-01,2024-04-01,25.00,100.0,200.0,125.00,500.0,1000.0
2878,3762,ED,carbidopa entacapone levodopa,5,2023-11-01,2026-01-16,31.25,125.0,200.0,156.25,625.0,1000.0
3048,3778,ED,carbidopa entacapone levodopa,4,2022-01-01,2024-03-01,25.00,100.0,200.0,100.00,400.0,800.0
...,...,...,...,...,...,...,...,...,...,...,...,...
7976,148492,ED,carbidopa entacapone levodopa,3,2023-07-01,2026-01-16,37.50,150.0,200.0,112.50,450.0,600.0
8023,151111,ED,carbidopa entacapone levodopa,3,2024-11-01,2026-01-16,50.00,200.0,200.0,150.00,600.0,600.0
8401,211045,ED,carbidopa entacapone levodopa,6,2023-03-01,2026-01-16,25.00,100.0,200.0,150.00,600.0,1200.0
8844,381012,ED,carbidopa entacapone levodopa,2,2018-01-01,2026-01-16,18.75,75.0,200.0,37.50,150.0,400.0


#### LEDD unificacion

In [68]:
# Lista de todos los dataframes
dfs = [LEDD_1_meds, LEDD_2_meds, LEDD_3_meds]

# Unimos usando concat con outer para incluir todas las columnas
LEDD_final = pd.concat(dfs, axis=0, join='outer').fillna(0)

print(LEDD_final.shape)
LEDD_final.head(3)
LEDD_final.drop(columns=['Frequency_per_day'],inplace=True)

LEDD_final

(5572, 36)


,PATNO,Visit ID,Medication,Start Date,Stop Date,Dose_Strength_mg_amantadine,Total_mg_per_day_amantadine,Dose_Strength_mg_selegiline,Total_mg_per_day_selegiline,Dose_Strength_mg_rotigotine,...,Dose_Strength_mg_safinamide,Total_mg_per_day_safinamide,Dose_Strength_mg_piribedil,Total_mg_per_day_piribedil,Dose_Strength_mg_carbidopa,Dose_Strength_mg_benserazide,Total_mg_per_day_carbidopa,Total_mg_per_day_benserazide,Dose_Strength_mg_entacapone,Total_mg_per_day_entacapone
0,3001,ED,amantadine,2022-03-01,2022-07-01,100.0,100.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0
1,3001,ED,amantadine,2020-07-01,2022-03-01,100.0,200.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0
2,3001,LOG,selegiline,2012-07-01,2012-08-01,0.0,0.0,5.0,50.0,0.0,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0
3,3001,LOG,rotigotine,2013-08-01,2013-09-01,0.0,0.0,0.0,0.0,5.0,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0
4,3001,LOG,amantadine,2018-07-01,2020-06-01,105.0,210.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7976,148492,ED,carbidopa entacapone levodopa,2023-07-01,2026-01-16,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,37.50,0.0,112.5,0.0,200.0,600.0
8023,151111,ED,carbidopa entacapone levodopa,2024-11-01,2026-01-16,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,50.00,0.0,150.0,0.0,200.0,600.0
8401,211045,ED,carbidopa entacapone levodopa,2023-03-01,2026-01-16,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,25.00,0.0,150.0,0.0,200.0,1200.0
8844,381012,ED,carbidopa entacapone levodopa,2018-01-01,2026-01-16,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,18.75,0.0,37.5,0.0,200.0,400.0


In [69]:
cols = ['PATNO', 'Visit ID', 'Start Date',	'Stop Date']

for col in ['Medication']+LEDD_final.filter(like="_mg_").columns.tolist():
    for i in range(12):
        cols.append(f"Month_{i}_{col}")

# Crear el DF vacío con todas las columnas de una vez
LEDD_info = pd.DataFrame(columns=cols)
LEDD_info

# esto lo hacemos porque depsues vamos a comprimirlo en consumo promedio 

,PATNO,Visit ID,Start Date,Stop Date,Month_0_Medication,Month_1_Medication,Month_2_Medication,Month_3_Medication,Month_4_Medication,Month_5_Medication,...,Month_2_Total_mg_per_day_entacapone,Month_3_Total_mg_per_day_entacapone,Month_4_Total_mg_per_day_entacapone,Month_5_Total_mg_per_day_entacapone,Month_6_Total_mg_per_day_entacapone,Month_7_Total_mg_per_day_entacapone,Month_8_Total_mg_per_day_entacapone,Month_9_Total_mg_per_day_entacapone,Month_10_Total_mg_per_day_entacapone,Month_11_Total_mg_per_day_entacapone


In [70]:
subjects_data=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_SUBJECT_CHARACTERISTICS_12SEP2025.csv',dtype=str)
subjects_data=subjects_data[['PATNO','Visit ID','DATE_VISIT']]
subjects_data['DATE_VISIT']=pd.to_datetime(subjects_data['DATE_VISIT'], errors='coerce')
subjects_data['NEXT_VISIT']=subjects_data['DATE_VISIT']+pd.DateOffset(months=12)

In [71]:
LEDD_info_list = []

for patno in subjects_data['PATNO'].unique():

    subject_visits = subjects_data.loc[subjects_data['PATNO']==patno, :]

    for visit in subject_visits['Visit ID'].unique():

        time_points = subject_visits.loc[subject_visits['Visit ID']==visit, ['DATE_VISIT','NEXT_VISIT']]

        start_date = time_points['DATE_VISIT'].iloc[0]
        stop_date  = time_points['NEXT_VISIT'].iloc[0]

        # Filter LEDD data for this subject
        data_filtered = LEDD_final.loc[LEDD_final['PATNO']==patno].copy()

        mask = (data_filtered['Start Date'] <= stop_date) & (data_filtered['Stop Date'] >= start_date)
        meds_in_visit = data_filtered.loc[mask, :]

        lista_df = []

        # Generate each month window
        for month in range(12):

            month_start = start_date + pd.DateOffset(months=month)
            month_end   = start_date + pd.DateOffset(months=month+1) - pd.DateOffset(days=1)

            mask_month = (meds_in_visit['Start Date'] <= month_end) & \
                         (meds_in_visit['Stop Date'] >= month_start)

            meds_in_month = meds_in_visit.loc[
                mask_month,
                ~meds_in_visit.columns.isin(['PATNO','Visit ID','Start Date','Stop Date'])
            ]

            if meds_in_month.empty:
                continue

            # Aggregate numeric and string columns
            resultado = {}
            for col in meds_in_month.columns:
                if meds_in_month[col].dtype == object:
                    resultado[col] = [list(meds_in_month[col].unique())]
                else:
                    resultado[col] = [meds_in_month[col].sum()]

            meds_visit = pd.DataFrame(resultado)

            # Prefix with Month label
            meds_visit = meds_visit.add_prefix(f"Month_{month}_")

            lista_df.append(meds_visit)

        if lista_df:
            final_df = pd.concat(lista_df, axis=1)
            final_df['PATNO'] = patno
            final_df['Visit ID'] = visit
            LEDD_info_list.append(final_df)

LEDD_info = pd.concat(LEDD_info_list, ignore_index=True)



In [72]:
LEDD_info.isna().sum()# este problema viene de los meses que no se consume ningun medicamento

# Columnas que contienen 'Medication'
cols_med = LEDD_info.filter(like="Medication").columns

# Columnas que NO contienen 'Medication'
cols_otros = LEDD_info.columns.difference(cols_med)

# Imputar vacío en las columnas de Medication
LEDD_info[cols_med] = LEDD_info[cols_med].fillna("")

# Imputar 0 en las demás columnas
LEDD_info[cols_otros] = LEDD_info[cols_otros].fillna(0)



In [73]:
LEDD_info.head(3)

,Month_11_Medication,Month_11_Dose_Strength_mg_amantadine,Month_11_Total_mg_per_day_amantadine,Month_11_Dose_Strength_mg_selegiline,Month_11_Total_mg_per_day_selegiline,Month_11_Dose_Strength_mg_rotigotine,Month_11_Total_mg_per_day_rotigotine,Month_11_Dose_Strength_mg_pramipexole,Month_11_Total_mg_per_day_pramipexole,Month_11_Dose_Strength_mg_ropinirole,...,Month_10_Dose_Strength_mg_safinamide,Month_10_Total_mg_per_day_safinamide,Month_10_Dose_Strength_mg_piribedil,Month_10_Total_mg_per_day_piribedil,Month_10_Dose_Strength_mg_carbidopa,Month_10_Dose_Strength_mg_benserazide,Month_10_Total_mg_per_day_carbidopa,Month_10_Total_mg_per_day_benserazide,Month_10_Dose_Strength_mg_entacapone,Month_10_Total_mg_per_day_entacapone
0,[trihexyphenidyl],0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,[trihexyphenidyl],0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,[trihexyphenidyl],0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [74]:
# ----------------------------
# IDENTIFICAR COLUMNAS CLAVE
# ----------------------------

# columnas de dosis y total
dose_cols = [c for c in LEDD_info.columns if "Dose_Strength_mg_" in c]
total_cols = [c for c in LEDD_info.columns if "Total_mg_per_day_" in c]
all_cols = dose_cols + total_cols

# columnas de medicación
med_cols = [c for c in LEDD_info.columns if "_Medication" in c]


# ----------------------------
# TRANSFORMAR A FORMATO LARGO
# ----------------------------

long_df = LEDD_info.melt(
    id_vars=["PATNO", "Visit ID"],
    value_vars=all_cols,
    var_name="variable",
    value_name="value"
)

# extraer medicamento y tipo
long_df["Medication"] = long_df["variable"].str.extract(
    r"(?:Dose_Strength_mg_|Total_mg_per_day_)(.*)"
)
long_df["Type"] = long_df["variable"].apply(
    lambda x: "Dose_Strength" if "Dose_Strength" in x else "Total_mg_per_day"
)


# ----------------------------
# CALCULAR MEDIA Y STD
# ----------------------------

stats = (
    long_df.groupby(["PATNO", "Visit ID", "Medication", "Type"])["value"]
    .agg(["mean", "std"])
    .reset_index()
)


# ----------------------------
# PIVOTEAR A FORMATO ANCHO
# ----------------------------

final_df = stats.pivot_table(
    index=["PATNO", "Visit ID"],
    columns=["Medication", "Type"],
    values=["mean", "std"]
)

final_df.columns = [
    f"{med}_{typ}_{stat}"
    for stat, med, typ in final_df.columns
]

final_df = final_df.reset_index()


# ----------------------------
# UNIÓN DE MEDICAMENTOS POR FILA
# ----------------------------

def union_medications(row):
    meds = set()
    for col in med_cols:
        val = row[col]

        # si es lista real
        if isinstance(val, list):
            meds.update(val)

        # si es string tipo "['levodopa','amantadine']"
        elif isinstance(val, str) and val.strip().startswith('['):
            try:
                parsed = eval(val)
                meds.update(parsed)
            except:
                pass
    return sorted(list(meds))  # ordenado opcional


# crear df con meds
med_df = LEDD_info[["PATNO", "Visit ID"] + med_cols].copy()
med_df["All_Medications"] = med_df.apply(union_medications, axis=1)
med_df = med_df[["PATNO", "Visit ID", "All_Medications"]]


# ----------------------------
# UNIR TODO
# ----------------------------

final_df = final_df.merge(med_df, on=["PATNO", "Visit ID"], how="left")


# ----------------------------
# FINAL: final_df
# ----------------------------

final_df.head()


,PATNO,Visit ID,amantadine_Dose_Strength_mean,amantadine_Total_mg_per_day_mean,apomorphine_Dose_Strength_mean,apomorphine_Total_mg_per_day_mean,benserazide_Dose_Strength_mean,benserazide_Total_mg_per_day_mean,carbidopa_Dose_Strength_mean,carbidopa_Total_mg_per_day_mean,...,rotigotine_Total_mg_per_day_std,safinamide_Dose_Strength_std,safinamide_Total_mg_per_day_std,selegiline_Dose_Strength_std,selegiline_Total_mg_per_day_std,trihexyphenidyl_Dose_Strength_std,trihexyphenidyl_Total_mg_per_day_std,zonisamide_Dose_Strength_std,zonisamide_Total_mg_per_day_std,All_Medications
0,100001,BL,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.57735,28.867513,0.0,0.0,[trihexyphenidyl]
1,100001,V04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.000000,0.0,0.0,[trihexyphenidyl]
2,100001,V06,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.000000,0.0,0.0,[trihexyphenidyl]
3,100001,V08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.000000,0.0,0.0,[trihexyphenidyl]
4,100001,V10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.000000,0.0,0.0,[trihexyphenidyl]


## Normal medication

In [75]:
med_df=pd.read_csv(
    '/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Medical History/Medical/DATA/Concomitant_Medication_Log_12Sep2025.csv',
    dtype=str)

med_df=decoder_DF(med_df, code_rows, code_cols, module='CONMED')
med_df=med_df[med_df['PATNO'].isin(PATNOs)]
print(med_df.shape)
med_df.columns=['Record ID', 'PATNO', 'Visit ID', 'Page Name', 'Medication', 'Dose',
       'Units', 'Frequency', 'Route', 'Start Date', 'Stop Date', 'Ongoing',
       'Indication1', 'Indication2', 'Total Daily Dose', 'WHO RECNO',
       'WHO SEQNO1', 'WHO SEQNO2', 'WHO DRUG NAME', 'Exclusionary Med flag',
       'Date of original data entry', 'Date of most recent update to record']

med_df['Indication1']=med_df['Indication1'].replace('NaN', np.nan)
med_df['Indication2']=med_df['Indication2'].replace('NaN', np.nan)
med_df['Indication1'] = med_df['Indication1'].fillna(med_df['Indication2'])
med_df.drop(columns=['Indication2'], inplace=True)

med_df['Indication1'] = med_df['Indication1'].astype(str).str.strip().str.upper()

        
med_df['Start Date'] = pd.to_datetime(med_df['Start Date'], errors='coerce')
med_df['Stop Date'] = pd.to_datetime(med_df['Stop Date'], errors='coerce')

med_df = med_df.dropna(subset=['Start Date'])
primer_dia_mes=pd.Timestamp.today().replace(day=1)

med_df.loc[med_df['Stop Date'].isna(), 'Stop Date'] = primer_dia_mes
lista_ind=med_df['Indication1'].unique().tolist()

with open("/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/DATA_CLEANING_PD/indicaciones_clasificadas_mejorado_v2.json", "r") as f:
    clasificacion = json.load(f)

# 3. Crear una nueva columna mapeada usando el JSON
med_df["Category"] = med_df["Indication1"].map(clasificacion)

med_df['Category'].value_counts(dropna=False)




(11943, 22)


/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col].replace('NaN', np.nan, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJ

Category
Otro / No clasificado         2971
Preventiva / Suplementos      2741
Cardiovascular                1910
Neurológica / Psiquiátrica    1625
Digestiva                      683
Endocrina / Metabólica         665
Musculoesquelética / Dolor     598
Urinaria / Genitourinaria      550
Infecciosa                      90
Dermatológica                   81
Ginecológica / Hormonales       16
Name: count, dtype: int64

In [76]:

def calculate_active_intervals(df):
    df = df.copy()
    patno = df['PATNO'].iloc[0]

    # Crear eventos de inicio y fin
    start_events = df[['Start Date']].rename(columns={'Start Date': 'Date'})
    stop_events = df[['Stop Date']].rename(columns={'Stop Date': 'Date'})

    # Marcar tipo de evento
    start_events['Event'] = 'Start'
    stop_events['Event'] = 'Stop'

    # Combinar y ordenar eventos
    events = pd.concat([start_events, stop_events], ignore_index=True)
    events = events.sort_values('Date').reset_index(drop=True)

    # Crear intervalos consecutivos
    intervals = pd.DataFrame({
        'Start Date': events['Date'].iloc[:-1].values,
        'Stop Date': events['Date'].iloc[1:].values
    })

    # Calcular categorías activas por intervalo
    active_category = []
    for _, row in intervals.iterrows():
        start, stop = row['Start Date'], row['Stop Date']
        category = (
            df[(df['Start Date'] <= start) & (df['Stop Date'] >= stop)]
            ['Category']
            .tolist()
        )
        active_category.append(' + '.join(sorted(category)))

    intervals['Active Category'] = active_category
    intervals['PATNO'] = patno

    # Retornar DataFrame con todas las columnas
    return intervals[['PATNO', 'Start Date', 'Stop Date', 'Active Category']]


med_df = (
    med_df.groupby('PATNO', group_keys=False)
          .apply(calculate_active_intervals)
          .reset_index(drop=True)
)


med_df.head()

/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_58251/3950140205.py:43: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calculate_active_intervals)


,PATNO,Start Date,Stop Date,Active Category
0,100001,2005-01-01,2010-01-01,Otro / No clasificado
1,100001,2010-01-01,2010-01-01,Cardiovascular + Otro / No clasificado + Otro ...
2,100001,2010-01-01,2014-01-01,Cardiovascular + Otro / No clasificado + Otro ...
3,100001,2014-01-01,2015-01-01,Cardiovascular + Otro / No clasificado + Otro ...
4,100001,2015-01-01,2015-01-01,Cardiovascular + Otro / No clasificado + Otro ...


In [77]:
med_df = med_df.loc[med_df['Start Date'] != med_df['Stop Date']]
med_df.head()


,PATNO,Start Date,Stop Date,Active Category
0,100001,2005-01-01,2010-01-01,Otro / No clasificado
2,100001,2010-01-01,2014-01-01,Cardiovascular + Otro / No clasificado + Otro ...
3,100001,2014-01-01,2015-01-01,Cardiovascular + Otro / No clasificado + Otro ...
5,100001,2015-01-01,2018-01-01,Cardiovascular + Otro / No clasificado + Otro ...
6,100001,2018-01-01,2019-01-01,Cardiovascular + Digestiva + Otro / No clasifi...


## Adverse Event In Clinic

In [78]:
# Cargar el archivo
AE_LOG_clinic = pd.read_csv(
    '/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Medical History/Medical/DATA/Adverse_Event_In-Clinic_Assessment_12Sep2025.csv',
    dtype=str
)

# Decodificar datos
AE_LOG_clinic = decoder_DF(AE_LOG_clinic, code_rows, code_cols, module='AECLINAST')

# Filtrar por participantes y visitas relevantes
AE_LOG_clinic = AE_LOG_clinic[AE_LOG_clinic['PATNO'].isin(PATNOs)]
AE_LOG_clinic = AE_LOG_clinic.loc[
    AE_LOG_clinic['Visit ID'].isin(["BL", "V04", "V06", "V08", "V10", "V12"]),
    :
]

# Definir columnas
list_cols_AE_general = [
    'Was an LP, skin biopsy, or dopamine imaging scan conducted at this visit?',
    'Were adverse events assessed following the procedure(s) on this assessment date?',
    'If Yes, were any adverse events observed?'
]

list_cols_AE_specific = [
    'LP performed on assessment date',
    'Skin Biopsy performed on assessment date',
    'Dopamine Imaging performed on assessment date'
]

# Crear máscaras lógicas
mask_no_proc = AE_LOG_clinic[list_cols_AE_general[0]] == 'No'

mask_2_proc = (
    (AE_LOG_clinic[list_cols_AE_general[0]] == 'Yes') &
    (AE_LOG_clinic[list_cols_AE_general[1]] == 'No')
)

mask_3_proc = (
    (AE_LOG_clinic[list_cols_AE_general[0]] == 'Yes') &
    (AE_LOG_clinic[list_cols_AE_general[1]] == 'Yes') &  # ← aquí había un error lógico
    (AE_LOG_clinic[list_cols_AE_general[2]] == 'No')
)

# Aplicar correcciones
AE_LOG_clinic.loc[mask_no_proc, list_cols_AE_general[1:]] = 'No'
AE_LOG_clinic.loc[mask_no_proc, list_cols_AE_specific] = 'Unchecked'

AE_LOG_clinic.loc[mask_2_proc, list_cols_AE_general[2]] = 'No'
AE_LOG_clinic.loc[mask_2_proc, list_cols_AE_specific] = 'Unchecked'

AE_LOG_clinic.loc[mask_3_proc, list_cols_AE_specific] = 'Unchecked'



/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)


In [79]:
AE_LOG_clinic=AE_LOG_clinic[['PATNO', 'Visit ID','If Yes, were any adverse events observed?'] + list_cols_AE_specific ]
AE_LOG_clinic.rename(columns={'If Yes, were any adverse events observed?':'Any adverse events observed?'}, inplace=True)
AE_LOG_clinic.isna().sum()
AE_LOG_clinic.drop_duplicates(subset=['PATNO','Visit ID'],inplace=True)
AE_LOG_clinic.dropna(how='any',inplace=True)
AE_LOG_clinic.isna().sum()

PATNO                                            0
Visit ID                                         0
Any adverse events observed?                     0
LP performed on assessment date                  0
Skin Biopsy performed on assessment date         0
Dopamine Imaging performed on assessment date    0
dtype: int64

In [80]:
AE_LOG_clinic

,PATNO,Visit ID,Any adverse events observed?,LP performed on assessment date,Skin Biopsy performed on assessment date,Dopamine Imaging performed on assessment date
161,40816,V12,No,Unchecked,Unchecked,Unchecked
188,42033,V10,No,Unchecked,Unchecked,Unchecked
189,42033,V12,No,Unchecked,Unchecked,Unchecked
191,42079,V10,No,Unchecked,Unchecked,Unchecked
192,42079,V12,No,Unchecked,Unchecked,Unchecked
...,...,...,...,...,...,...
9848,431490,BL,No,Unchecked,Unchecked,Unchecked
9893,446160,BL,No,Unchecked,Unchecked,Unchecked
9897,446733,BL,No,Unchecked,Unchecked,Unchecked
9922,476236,BL,No,Unchecked,Unchecked,Unchecked


## Adverse events out of clinic

In [81]:
AE_log=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Medical History/Safty Monitoring/DATA/Adverse_Event_Log_12Sep2025.csv', dtype=str)
AE_log=decoder_DF(AE_log, code_rows, code_cols, module='AE')
AE_log=AE_log[AE_log['PATNO'].isin(PATNOs)]
AE_log

/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)


,Record ID,PATNO,Visit ID,Page Name,Site Aware Date,Adverse Event,Start Date,Stop Date,Severity,SAE,...,Outcome,Low Level Term,Preferred Term Code,Preferred Term Name,High Level Term,High Level Group Term,System Organ Class 1,MedDRA Version,Date of original data entry,Date of most recent update to record
2,288616401,3001,LOG,Adverse Event Log,NaN,LOW BACK DISCOMFORT RELATED TO LP,03/2011,03/2011,Mild,No,...,Recovered,Back discomfort,10053156,Post-LP Back Pain,Musculoskeletal and connective tissue pain and...,Musculoskeletal and connective tissue disorder...,Musc,16.1,04/2011,2020-06-25 16:04:31.0
3,288616501,3001,LOG,Adverse Event Log,NaN,STIFF NECK,03/2011,03/2011,Mild,No,...,Recovered,Stiff neck,10052904,Musculoskeletal stiffness,Musculoskeletal and connective tissue signs an...,Musculoskeletal and connective tissue disorder...,Musc,16.1,04/2011,2020-06-25 16:04:31.0
5,290335301,3003,LOG,Adverse Event Log,NaN,HEADACHE,04/2011,04/2011,Mild,No,...,Recovered,Headache,10019211,Post-LP Headache,Headaches NEC,Headaches,Nerv,16.1,05/2011,2020-06-25 16:06:24.0
6,290335401,3003,LOG,Adverse Event Log,NaN,BACKACHE,04/2011,04/2011,Mild,No,...,Recovered,Backache,10003988,Post-LP Back Pain,Musculoskeletal and connective tissue pain and...,Musculoskeletal and connective tissue disorder...,Musc,16.1,05/2011,2020-06-25 16:06:24.0
7,290335501,3003,LOG,Adverse Event Log,NaN,NECK STIFFNESS,04/2011,04/2011,Mild,No,...,Recovered,Neck stiffness,10052904,Musculoskeletal stiffness,Musculoskeletal and connective tissue signs an...,Musculoskeletal and connective tissue disorder...,Musc,16.1,05/2011,2020-06-25 16:06:24.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2867,IA556315,357613,ED,Adverse Event Log,08/2024,Neck pain and occipital headache worse in the ...,08/2024,08/2024,Moderate,No,...,Recovered,NaN,NaN,NaN,NaN,NaN,NaN,NaN,08/2024,2024-09-01 00:00:00.0
2970,IA653748,407230,ED,Adverse Event Log,01/2025,Headache,01/2025,NaN,Mild,No,...,Under treatment / observation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,01/2025,2025-01-13 00:00:00.0
3000,IA785025,423325,ED,Adverse Event Log,07/2025,Headache after lumbar puncture,07/2025,07/2025,Mild,No,...,Recovered,NaN,NaN,NaN,NaN,NaN,NaN,NaN,07/2025,2025-07-16 00:00:00.0
3005,IA746535,425401,ED,Adverse Event Log,05/2025,Mild headache,05/2025,NaN,Mild,No,...,Under treatment / observation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,05/2025,2025-05-27 00:00:00.0


## Neurological Exam

In [82]:
neuro_df=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Medical History/Neurological Exam/DATA/Neurological_Exam_12Sep2025.csv', dtype=str)
neuro_df=decoder_DF(neuro_df, code_rows, code_cols, module='PENEURO')
neuro_df=neuro_df[neuro_df['PATNO'].isin(PATNOs)]
neuro_df=neuro_df.loc[neuro_df['Visit ID'].isin(["BL", "V04", "V06", "V08", "V10", "V12"]),:]
neuro_df.isna().sum()
neuro_col=['PATNO', 'Visit ID', 'Motor Exam assessment', 'Coordination assessment',
       'Sensory Exam assessment', 'Reflexes assessment','CN II-XII assessment']

neuro_df=neuro_df[neuro_col]
neuro_df.isna().sum()
neuro_df.drop_duplicates(subset=['PATNO','Visit ID'],inplace=True)
neuro_df.loc[neuro_df['Visit ID']=='BL','Visit ID']='V04'
neuro_df.isna().sum()

/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)


PATNO                      0
Visit ID                   0
Motor Exam assessment      0
Coordination assessment    0
Sensory Exam assessment    0
Reflexes assessment        0
CN II-XII assessment       0
dtype: int64

## Clinical Diagnosis Not Relevant

In [83]:
clin_diag_df=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Medical History/Medical/DATA/Clinical_Diagnosis_12Sep2025.csv', dtype=str)
clin_diag_df=decoder_DF(clin_diag_df, code_rows, code_cols, module='NEWCLINDX')
clin_diag_df=clin_diag_df.loc[clin_diag_df['Visit ID'].isin(["BL", "V04", "V06", "V08", "V10", "V12"]),:]
clin_diag_df=clin_diag_df[clin_diag_df['PATNO'].isin(PATNOs)]
clin_diag_df=clin_diag_df.loc[clin_diag_df['Indicate if diagnosis occurred between the last study visit and this study visit']=='No',:]
clin_relevant_cols=['PATNO', 'Visit ID', 'Most likely clinical diagnosis']
clin_diag_df=clin_diag_df[clin_relevant_cols]
clin_diag_df['Visit ID'].value_counts()
clin_diag_df.drop_duplicates(subset=['PATNO','Visit ID'],inplace=True)
clin_diag_df


/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)


,PATNO,Visit ID,Most likely clinical diagnosis
1330,42033,V12,Idiopathic PD
1333,42034,V12,Idiopathic PD
1341,42079,V12,Idiopathic PD
1478,43082,V12,Idiopathic PD
1480,43083,V12,Idiopathic PD
...,...,...,...
10738,320651,V04,Idiopathic PD
10800,324862,V04,Idiopathic PD
10804,325051,V04,Idiopathic PD
10819,325566,V04,Idiopathic PD


# ANALYSIS DE PROGRESSION

In [84]:
progression_csv_upgrade(vital_df,prefix='VITAL')
progression_multi_csv_upgrade(vital_df,prefix='VITAL')
visit_csv_upgrade(vital_df,prefix='VITAL')


progression_csv_upgrade(neuro_df,prefix='NEURO')
progression_multi_csv_upgrade(neuro_df,prefix='NEURO')
visit_csv_upgrade(neuro_df,prefix='NEURO')


/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:59: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  p1.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:65: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  p2.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:71: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `

Progression CSV files updated successfully.
MULTI Progression CSV files updated successfully.
Visits CSV files updated successfully.


/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:265: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pV06.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:271: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pV08.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:277: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior

Progression CSV files updated successfully.
MULTI Progression CSV files updated successfully.
Visits CSV files updated successfully.


/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:193: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  p1.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:199: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  p2.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:205: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, se

In [85]:
cols_asignacion2(vital_df,df_secundario='VITAL', df_main='MEDICAL')
cols_asignacion2(neuro_df,df_secundario='NEURO', df_main='MEDICAL')

,df_main,df_secundario,df_secundario_col
0,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,PATNO
1,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,Sporadic PD at Enrollment
2,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,RBD at Enrollment
3,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,Pink1 Mutation at Enrollment
4,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,Parkin Mutation at Enrollment
...,...,...,...
141,MEDICAL,NEURO,Motor Exam assessment
142,MEDICAL,NEURO,Coordination assessment
143,MEDICAL,NEURO,Sensory Exam assessment
144,MEDICAL,NEURO,Reflexes assessment


In [86]:
vital_df.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_VITAL_12SEP2025.csv', index=False)
neuro_df.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_NEURO_12SEP2025.csv', index=False)

# SPECIAL ANALYSIS 

In [87]:
info_visit=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_SUBJECT_CHARACTERISTICS_12SEP2025.csv', dtype=str)
info_visit=info_visit[info_visit['PATNO'].isin(PATNOs)]
info_visit=info_visit[['PATNO', 'Visit ID','DATE_VISIT']]
info_visit['Visit ID'] = pd.Categorical(info_visit['Visit ID'], categories=orden_visitas, ordered=True)
info_visit=info_visit.sort_values(by=['PATNO', 'Visit ID']).reset_index(drop=True)
info_visit['DATE_VISIT'] = pd.to_datetime(info_visit['DATE_VISIT'], errors='coerce')
info_visit.head()

,PATNO,Visit ID,DATE_VISIT
0,100001,BL,2020-10-01
1,100001,V04,2021-11-01
2,100001,V06,2022-11-01
3,100001,V08,2023-11-01
4,100001,V10,2024-09-01


## AE_LOG_CLINIC

In [88]:
AE_LOG_clinic_new=AE_LOG_clinic.merge(info_visit, on=['PATNO', 'Visit ID'], how='right')


In [89]:
AE_LOG_clinic_new['Any adverse events observed?'] = (
    AE_LOG_clinic_new['Any adverse events observed?'].fillna('No')
)


AE_LOG_clinic_new.drop(columns=['DATE_VISIT'], inplace=True)


cols = [
    'LP performed on assessment date',
    'Skin Biopsy performed on assessment date',
    'Dopamine Imaging performed on assessment date'
]


AE_LOG_clinic_new.loc[:, cols] = AE_LOG_clinic_new.loc[:, cols].fillna('Unchecked')
print(AE_LOG_clinic_new.isna().sum())


PATNO                                            0
Visit ID                                         0
Any adverse events observed?                     0
LP performed on assessment date                  0
Skin Biopsy performed on assessment date         0
Dopamine Imaging performed on assessment date    0
dtype: int64


In [90]:
AE_LOG_clinic_new

,PATNO,Visit ID,Any adverse events observed?,LP performed on assessment date,Skin Biopsy performed on assessment date,Dopamine Imaging performed on assessment date
0,100001,BL,No,Unchecked,Unchecked,Unchecked
1,100001,V04,No,Unchecked,Unchecked,Unchecked
2,100001,V06,No,Unchecked,Unchecked,Unchecked
3,100001,V08,No,Unchecked,Unchecked,Unchecked
4,100001,V10,No,Unchecked,Unchecked,Unchecked
...,...,...,...,...,...,...
6325,75562,V04,No,Unchecked,Unchecked,Unchecked
6326,75562,V06,No,Unchecked,Unchecked,Unchecked
6327,75562,V08,No,Unchecked,Unchecked,Unchecked
6328,75562,V10,No,Unchecked,Unchecked,Unchecked


## AE_log out of clinic

In [91]:
AE_log['PATNO'].nunique()

405

In [92]:
AE_log.isna().sum()

Record ID                                 0
PATNO                                     0
Visit ID                                  0
Page Name                                 0
Site Aware Date                         350
Adverse Event                             0
Start Date                                0
Stop Date                                26
Severity                                  0
SAE                                       0
Relationship to Study                     0
Relationship to Study Procedure          52
Premature Withdrawal due to AE          352
Outcome                                   8
Low Level Term                          370
Preferred Term Code                     370
Preferred Term Name                     370
High Level Term                         370
High Level Group Term                   370
System Organ Class 1                    370
MedDRA Version                          370
Date of original data entry               0
Date of most recent update to re

In [93]:
AE_log['Relationship to Study'].unique()

array(['Definite', 'Probable', 'Possible', 'Unrelated', 'Unlikely'],
      dtype=object)

In [94]:
AE_log = AE_log.loc[
    (AE_log['Relationship to Study Procedure'] != 'PI-2620 PET (Tau PET Imaging)') &
    (AE_log['Relationship to Study'].isin(['Definite', 'Probable', 'Possible']))
]

AE_log.isna().sum()


Record ID                                 0
PATNO                                     0
Visit ID                                  0
Page Name                                 0
Site Aware Date                         298
Adverse Event                             0
Start Date                                0
Stop Date                                20
Severity                                  0
SAE                                       0
Relationship to Study                     0
Relationship to Study Procedure           1
Premature Withdrawal due to AE          300
Outcome                                   5
Low Level Term                          351
Preferred Term Code                     351
Preferred Term Name                     351
High Level Term                         351
High Level Group Term                   351
System Organ Class 1                    351
MedDRA Version                          351
Date of original data entry               0
Date of most recent update to re

In [95]:

AE_log['Start Date'] = pd.to_datetime(AE_log['Start Date'], errors='coerce')
AE_log['Stop Date'] = pd.to_datetime(AE_log['Stop Date'], errors='coerce')
hoy = pd.Timestamp(datetime.today().strftime('%Y-%m-01'))
# Rellenar los NaN con año y mes actual
AE_log['Stop Date'].fillna(hoy, inplace=True)


/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_58251/4101018749.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  AE_log['Start Date'] = pd.to_datetime(AE_log['Start Date'], errors='coerce')
/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_58251/4101018749.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  AE_log['Start Date'] = pd.to_datetime(AE_log['Start Date'], errors='coerce')
/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_58251/4101018749.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure pars

In [96]:
relevant_cols=['PATNO', 'Visit ID','Adverse Event','Severity','Relationship to Study','Relationship to Study Procedure','System Organ Class 1','Start Date','Stop Date']
AE_log=AE_log[relevant_cols]
AE_log['Relationship to Study Procedure'].fillna('Lumbar Puncture (PPMI Clinical)', inplace=True)
AE_log


/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_58251/1402722499.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  AE_log['Relationship to Study Procedure'].fillna('Lumbar Puncture (PPMI Clinical)', inplace=True)


,PATNO,Visit ID,Adverse Event,Severity,Relationship to Study,Relationship to Study Procedure,System Organ Class 1,Start Date,Stop Date
2,3001,LOG,LOW BACK DISCOMFORT RELATED TO LP,Mild,Definite,Lumbar Puncture (PPMI Clinical),Musc,2011-03-01,2011-03-01
3,3001,LOG,STIFF NECK,Mild,Probable,Lumbar Puncture (PPMI Clinical),Musc,2011-03-01,2011-03-01
5,3003,LOG,HEADACHE,Mild,Probable,Lumbar Puncture (PPMI Clinical),Nerv,2011-04-01,2011-04-01
6,3003,LOG,BACKACHE,Mild,Definite,Lumbar Puncture (PPMI Clinical),Musc,2011-04-01,2011-04-01
7,3003,LOG,NECK STIFFNESS,Mild,Possible,Lumbar Puncture (PPMI Clinical),Musc,2011-04-01,2011-04-01
...,...,...,...,...,...,...,...,...,...
2867,357613,ED,Neck pain and occipital headache worse in the ...,Moderate,Definite,Lumbar Puncture (PPMI Clinical),NaN,2024-08-01,2024-08-01
2970,407230,ED,Headache,Mild,Probable,Lumbar Puncture (PPMI Clinical),NaN,2025-01-01,2026-01-01
3000,423325,ED,Headache after lumbar puncture,Mild,Definite,Lumbar Puncture (PPMI Clinical),NaN,2025-07-01,2025-07-01
3005,425401,ED,Mild headache,Mild,Possible,Lumbar Puncture (PPMI Clinical),NaN,2025-05-01,2026-01-01


In [97]:
info_session=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_SUBJECT_CHARACTERISTICS_12SEP2025.csv', dtype=str)
info_session=info_session[info_session['PATNO'].isin(PATNOs)]
info_session=info_session[['PATNO', 'Visit ID','DATE_VISIT']]
info_session['Visit ID'] = pd.Categorical(info_session['Visit ID'], categories=orden_visitas, ordered=True)
info_session=info_session.sort_values(by=['PATNO', 'Visit ID']).reset_index(drop=True)
info_session.head()
info_session['DATE_VISIT'] = pd.to_datetime(info_session['DATE_VISIT'], errors='coerce')

In [98]:
def AE_ASSIGNATION(row):
    mask = (
        (AE_log['PATNO'] == row['PATNO']) &
        (AE_log['Start Date'] <= row['DATE_VISIT']) &
        (AE_log['Stop Date'] >= row['DATE_VISIT'])
    )
    subset = AE_log.loc[mask]
    
    if subset.empty:
        row['Total_AE'] = 0
        row['Mild_AE'] = 0
        row['Moderate_AE'] = 0
        row['Severe_AE'] = 0
        row['R_Definite_AE'] = 0
        row['R_Probable_AE'] = 0
        row['R_Possible'] = 0
        row['R_LP'] = 0
        row['R_DATscan'] = 0
        row['R_BiopsySkin'] = 0
    else:
        row['Total_AE'] = len(subset)
        row['Mild_AE'] = len(subset[subset['Severity'] == 'Mild'])
        row['Moderate_AE'] = len(subset[subset['Severity'] == 'Moderate'])
        row['Severe_AE'] = len(subset[subset['Severity'] == 'Severe'])
        row['R_Definite_AE'] = len(subset[subset['Relationship to Study'] == 'Definite'])
        row['R_Probable_AE'] = len(subset[subset['Relationship to Study'] == 'Probable'])
        row['R_Possible'] = len(subset[subset['Relationship to Study'] == 'Possible'])
        row['R_LP'] = len(subset[subset['Relationship to Study Procedure'].str.contains('Lumbar Puncture', na=False)])
        row['R_DATscan'] = len(subset[subset['Relationship to Study Procedure'].str.contains('DATSCAN', na=False)])
        row['R_BiopsySkin'] = len(subset[subset['Relationship to Study Procedure'].str.contains('Skin Biopsy', na=False)])
    
    return row

info_AE_sesion = info_session.apply(AE_ASSIGNATION, axis=1)
info_AE_sesion.isna().sum()


PATNO            0
Visit ID         0
DATE_VISIT       0
Total_AE         0
Mild_AE          0
Moderate_AE      0
Severe_AE        0
R_Definite_AE    0
R_Probable_AE    0
R_Possible       0
R_LP             0
R_DATscan        0
R_BiopsySkin     0
dtype: int64

In [99]:
info_AE_sesion.drop(columns=['DATE_VISIT','R_DATscan'], inplace=True)

In [100]:
info_AE_sesion.head()

,PATNO,Visit ID,Total_AE,Mild_AE,Moderate_AE,Severe_AE,R_Definite_AE,R_Probable_AE,R_Possible,R_LP,R_BiopsySkin
0,100001,BL,0,0,0,0,0,0,0,0,0
1,100001,V04,0,0,0,0,0,0,0,0,0
2,100001,V06,0,0,0,0,0,0,0,0,0
3,100001,V08,0,0,0,0,0,0,0,0,0
4,100001,V10,0,0,0,0,0,0,0,0,0


# Normal medication

In [101]:
from collections import Counter

# --- Convertir fechas ---
for col in ['Start Date', 'Stop Date']:
    med_df[col] = pd.to_datetime(med_df[col])

# --- Función para obtener medicaciones activas en cada visita ---
def meds_activas(row):
    mask = (
        (med_df['PATNO'] == row['PATNO']) &
        (med_df['Start Date'] <= row['DATE_VISIT']) &
        (med_df['Stop Date'] >= row['DATE_VISIT'])
    )
    subset = med_df.loc[mask]
    
    if subset.empty:
        return 'Ninguna'
    
    # Concatenar los nombres de medicaciones activas si hay más de una
    meds = ' + '.join(subset['Active Category'])
    return meds

# --- Aplicar la función ---
info_visit['Active Category'] = info_visit.apply(meds_activas, axis=1)

# --- Mantener columnas principales ---
med_info_visit = info_visit[['PATNO', 'Visit ID', 'DATE_VISIT', 'Active Category']]

# --- Crear columnas separadas por categoría con el número de apariciones ---
# Si la fila tiene 'Ninguna', se reemplaza por cadena vacía para evitar errores
expanded = med_info_visit['Active Category'].replace('Ninguna', '').apply(
    lambda x: Counter([cat.strip() for cat in x.split('+') if cat.strip() != ''])
)

# Convertir lista de contadores en DataFrame y unir con el original
expanded_df = pd.DataFrame(list(expanded)).fillna(0).astype(int)
med_info_visit_final = pd.concat([med_info_visit, expanded_df], axis=1)

# --- Ver resultado ---
med_info_visit_final.head()


,PATNO,Visit ID,DATE_VISIT,Active Category,Cardiovascular,Digestiva,Otro / No clasificado,Preventiva / Suplementos,Neurológica / Psiquiátrica,Endocrina / Metabólica,Urinaria / Genitourinaria,Musculoesquelética / Dolor,Dermatológica,Infecciosa,Ginecológica / Hormonales
0,100001,BL,2020-10-01,Cardiovascular + Digestiva + Digestiva + Otro ...,1,2,4,4,0,0,0,0,0,0,0
1,100001,V04,2021-11-01,Cardiovascular + Digestiva + Digestiva + Otro ...,1,2,4,4,0,0,0,0,0,0,0
2,100001,V06,2022-11-01,Cardiovascular + Digestiva + Digestiva + Neuro...,1,2,5,4,1,0,0,0,0,0,0
3,100001,V08,2023-11-01,Cardiovascular + Digestiva + Digestiva + Neuro...,1,2,5,4,1,0,0,0,0,0,0
4,100001,V10,2024-09-01,Cardiovascular + Digestiva + Digestiva + Neuro...,1,2,5,4,1,0,0,0,0,0,0


In [102]:
med_info_visit_final.loc[med_info_visit_final['Active Category']=='Otro / No clasificado','Active Category']='Ninguna'
med_info_visit_final.drop(columns=['Otro / No clasificado'], inplace=True)

def no_PD_medication(x):
    if x == 'Ninguna':
        return 0
    else:
        return 1

med_info_visit_final['NO PD Medication'] = med_info_visit_final['Active Category'].apply(no_PD_medication)
med_info_visit_final.drop(columns=['Active Category','DATE_VISIT'], inplace=True)
med_info_visit_final.head()

,PATNO,Visit ID,Cardiovascular,Digestiva,Preventiva / Suplementos,Neurológica / Psiquiátrica,Endocrina / Metabólica,Urinaria / Genitourinaria,Musculoesquelética / Dolor,Dermatológica,Infecciosa,Ginecológica / Hormonales,NO PD Medication
0,100001,BL,1,2,4,0,0,0,0,0,0,0,1
1,100001,V04,1,2,4,0,0,0,0,0,0,0,1
2,100001,V06,1,2,4,1,0,0,0,0,0,0,1
3,100001,V08,1,2,4,1,0,0,0,0,0,0,1
4,100001,V10,1,2,4,1,0,0,0,0,0,0,1


# Analysis Progression 2

In [103]:
progression_csv_upgrade(AE_LOG_clinic_new,prefix='AE_CLINIC')
progression_multi_csv_upgrade(AE_LOG_clinic_new,prefix='AE_CLINIC')
visit_csv_upgrade(AE_LOG_clinic_new,prefix='AE_CLINIC')
AE_LOG_clinic_new.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_AE_CLINIC_12SEP2025.csv', index=False)
cols_asignacion2(AE_LOG_clinic_new,df_secundario='AE_CLINIC', df_main='MEDICAL')

progression_csv_upgrade(info_AE_sesion,prefix='AE_NO_CLINIC')
progression_multi_csv_upgrade(info_AE_sesion,prefix='AE_NO_CLINIC')
visit_csv_upgrade(info_AE_sesion,prefix='AE_NO_CLINIC')
info_AE_sesion.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_AE_NO_CLINIC_12SEP2025.csv', index=False)
cols_asignacion2(info_AE_sesion,df_secundario='AE_NO_CLINIC', df_main='MEDICAL')

progression_csv_upgrade(final_df,prefix='LEDD_MEDS')
progression_multi_csv_upgrade(final_df,prefix='LEDD_MEDS')
visit_csv_upgrade(final_df,prefix='LEDD_MEDS')
final_df.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_LEDD_MEDS_12SEP2025.csv', index=False)
cols_asignacion2(final_df,df_secundario='LEDD_MEDS', df_main='MEDICAL')

progression_csv_upgrade(med_info_visit_final,prefix='NOT_PD_MEDS')
progression_multi_csv_upgrade(med_info_visit_final,prefix='NOT_PD_MEDS')
visit_csv_upgrade(med_info_visit_final,prefix='NOT_PD_MEDS')
med_info_visit_final.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_NOT_PD_MEDS_12SEP2025.csv', index=False)
cols_asignacion2(med_info_visit_final,df_secundario='NOT_PD_MEDS', df_main='MEDICAL')

Progression CSV files updated successfully.
MULTI Progression CSV files updated successfully.
Visits CSV files updated successfully.
Progression CSV files updated successfully.
MULTI Progression CSV files updated successfully.
Visits CSV files updated successfully.


/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:59: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  p1.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:65: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  p2.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:71: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `

Progression CSV files updated successfully.
MULTI Progression CSV files updated successfully.
Visits CSV files updated successfully.


/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:271: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pV08.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:277: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pV10.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:283: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior

Progression CSV files updated successfully.
MULTI Progression CSV files updated successfully.
Visits CSV files updated successfully.


,df_main,df_secundario,df_secundario_col
0,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,PATNO
1,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,Sporadic PD at Enrollment
2,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,RBD at Enrollment
3,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,Pink1 Mutation at Enrollment
4,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,Parkin Mutation at Enrollment
...,...,...,...
234,MEDICAL,NOT_PD_MEDS,Musculoesquelética / Dolor
235,MEDICAL,NOT_PD_MEDS,Dermatológica
236,MEDICAL,NOT_PD_MEDS,Infecciosa
237,MEDICAL,NOT_PD_MEDS,Ginecológica / Hormonales
